In [34]:
# Import your libraries here

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import yfinance as yf
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from edgar import *
from lxml import html
import asyncio
import aiohttp
import time
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
from transformers import logging as hf_logging
from concurrent.futures import ThreadPoolExecutor

In [2]:
current_dir = os.path.abspath("")
project_root = os.path.abspath(os.path.join(current_dir, "../"))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: D:\users\kamen.dimitrov\desktop\softuni\AI_and_ML_upskill_program\Data Science\08_final_project


In [3]:
# Import your modules here

import importlib

import src.plotting_utils.plotting_utils as plot_utils
from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.nlp_utils.nlp_utils as nlp_utils

importlib.reload(plot_utils)
importlib.reload(data_pipe)
importlib.reload(nlp_utils)

<module 'src.nlp_utils.nlp_utils' from 'D:\\users\\kamen.dimitrov\\desktop\\softuni\\AI_and_ML_upskill_program\\Data Science\\08_final_project\\src\\nlp_utils\\nlp_utils.py'>

### Introduction and goal of this notebook

In this notebook, I will establish the process of researching and obtaining the second dataset. It will include text samples of the earnings press releases from earning announcement 8-K type reports and the management discussion and analysis (MD&A) sections of quarterly earning 10-Q type reports. 

**N.B.!** After I experienced the initial project setback by realizing it is not feasible to obtain actual earnings calls transcripts without breaching ethical and legal boundaries of data aggregators, I believe this is going to be the second best option. I will make a more detailed discussion on the possible presence of **Management Bias** in the text samples selected via this method and the possibility of it contaminating the scientific study and leading to inconclusive results from statistical standpoint.

**This notebook is optional** as I will directly provide the final dataset after all data handling and extraction operations. You can observe, or test, or directly run the notebook to reproduce the results.

The process includes:
- Obtain a history of filings for the group of selected companies
- Convert filing date to Pandas datetime
- Filter the required sample - in my case I will focus on filings after 2016-03-31, which currently is the last 10 years
- Use the Data column in the Data Frame to filter the 8-K reports, which contain earnings
- Scrape the EDGAR database to confirm that the ITEMS field on this record contains Item 2.02, which is the only indicator we have an 8-K report we need
- Scrape the EDGAR database to obtain Exhibit 99.1 press releases from the filtered 8-K reports and the MD&A sections from the 10-Q reports
- Verify I now have a full working dataset
- Utilize the services of Data Version Control (DVC) as the Data Set will likely be hundreds of megabites.

I would like to emphasize again that this notebook is only proof of concept and source of data obtaining and it remains optional to implement, as the next notebook will utilize the final product via DVC.

**Source of preliminary information** - [Calcbench filing filter](https://www.calcbench.com/filings?pg_classificationMethod=tickers&pg_tickers=AMD,GOOG,AAPL,T,BAC,CAT,CVX,CB,DE,XOM,FE,F,INTC,JNJ,JPM,LMT,MCD,MSFT,NKE,NVDA,ORCL,PFE,PG,CRM,TGT,UPS,VZ,V,WMT,DIS&pc_rangeOption=All%20History&f_filingTypes=BusinessWirePR_filedAfterAn8K&f_filingTypes=BusinessWirePR_replaced&f_filingTypes=proxy&f_filingTypes=annualQuarterlyReport&f_filingTypes=eightk_earningsPressRelease&f_filingTypes=eightk_guidanceUpdate&f_filingTypes=eightk_presentationSlides&f_filingTypes=eightk_monthlyOperatingMetrics&f_filingTypes=eightk_earningsPressRelease_preliminary&f_filingTypes=eightk_earningsPressRelease_correction&f_filingTypes=eightk_other&f_filingTypes=commentLetter&f_filingTypes=commentLetterResponse&f_filingTypes=eightk_nonfinancial&f_filingTypes=NT10KorQ&f_filingTypes=S&f_filingTypes=Four24B&f_filingTypes=institutionalOwnsership_13F&f_filingTypes=ForeignAnnualOrInterimReport) - Exported from Calcbench on 2026-03-31. 

Unfortunately, I have access to Calcbench through my CFA Institute member benefits. However, this particular calbench service appears to be generally part of the free tier service provided by them. The end goal is to actually have a list of tickers and to extract all history of filings these public companies committed to the SEC under legal requirements. 

It appears that clicking the above source, which requires no authentication, opens the most recent data extraction result. Calcbench is a legitimate and reputable data service provider, so I hope that the above link will remain functional for a protracted period of time and produce actual and current data for the selected group of stocks. 

What calcbench does for me is providing the date of the filing and the link as a url saved in a string format to the SEC's EDGAR database. This simply saves several hours of manual work in EDGAR itself.

In my case, this historical extract will be directly read from the static data folder and faterwards I will perform and explain the logic behind several data handling operations. This will narrow down the rows to under 2000 and afterwards, I will use web scraping techniques to collect the text data and store in a file, which is going to be source of data in the actual academic exercise in the following notebook. 

#### 1. So, let's extract the static data in a Data Frame

In [23]:
# As a reminder I will declare the tickers variable as in the starting workbook 1_1
tickers = [
    "AAPL", "GOOG", "MSFT", "NVDA",
    "JPM", "BAC", "F", "UPS", "WMT", "TGT",
    "VZ", "T", "FE", "PFE", "JNJ", "DIS",
    "V", "MCD", "NKE", "XOM", "CVX",
    "CAT", "DE", "LMT", "AMD", "INTC", "ORCL", 
    "CRM", "CB", "PG"
]

# Extract the data from the calcbench CSV file
filing_data = pd.read_csv("static_data/calcbench_filings.csv", sep=';')
filing_data

,Filing ID,Company,Ticker,CIK,Filing Date,Filing Type,Period End,Fiscal Year,Fiscal Period,SEC Link,Calcbench Published,Data
0,2575776,ADVANCED MICRO DEVICES INC,AMD,2488,26.3.2026,DEF 14A,26.12.2025,2025.0,4.0,https://www.sec.gov/Archives/edgar/data/2488/0...,27.3.2026,False
1,2575712,Walmart Inc.,WMT,104169,26.3.2026,8-K,29.4.2026,2027.0,1.0,https://www.sec.gov/Archives/edgar/data/104169...,27.3.2026,False
2,2575404,FORD MOTOR CO,F,37996,26.3.2026,DEF 14A,30.12.2025,2025.0,4.0,https://www.sec.gov/Archives/edgar/data/37996/...,27.3.2026,False
3,2575170,CATERPILLAR INC,CAT,18230,25.3.2026,8-K,30.3.2026,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/18230/...,26.3.2026,False
4,2574747,LOCKHEED MARTIN CORP,LMT,936468,25.3.2026,DEF 14A,30.12.2025,2025.0,4.0,https://www.sec.gov/Archives/edgar/data/936468...,26.3.2026,False
...,...,...,...,...,...,...,...,...,...,...,...,...
10946,1446319,Visa Inc.,V,1403161,3.2.2008,10-Q,30.12.2007,2008.0,1.0,https://www.sec.gov/Archives/edgar/data/140316...,4.2.2008,False
10947,1451590,PROCTER & GAMBLE Co,PG,80424,31.1.2008,10-Q,30.12.2007,2008.0,2.0,https://www.sec.gov/Archives/edgar/data/80424/...,1.2.2008,False
10948,1448087,Apple Inc,AAPL,320193,31.1.2008,10-Q,28.12.2007,2008.0,1.0,https://www.sec.gov/Archives/edgar/data/320193...,1.2.2008,False
10949,1451223,Microsoft Corp,MSFT,789019,23.1.2008,10-Q,30.12.2007,2008.0,2.0,https://www.sec.gov/Archives/edgar/data/789019...,24.1.2008,False


In [ ]:
filing_data.shape

#### 2. Convert the date from a string format ot a Pandas datetime format

In [ ]:
filing_data['Filing Date'] = pd.to_datetime(filing_data['Filing Date'], errors='coerce')
filing_data['Calcbench Published'] = pd.to_datetime(filing_data['Calcbench Published'], errors='coerce')
filing_data.info()

In [33]:
def extract_mdna_text(accession):
    set_identity("kamendd@hotmail.com")   
    filing = find(accession)
    mda_text = filing.obj()["Item 2"]
    if mda_text:
        return mda_text
    else:
        return pd.NA

def extract_press_release(accession):
    set_identity("kamendd@hotmail.com")
    filing = find(accession)
    eight_k = filing.obj()
    if eight_k.has_press_release:
        releases = eight_k.press_releases
        pr = releases[0]
        return pr.text()
    else:
        return pd.NA

def extract_accession_number(url):
    tokens = url.split('/')
    accession_number = tokens[-1].replace('-index.htm', '')
    return accession_number

def process_filing_row(row):
    accession = row['accession']
    filing_type = row['Filing Type']
    text = None
    
    if filing_type == '10-Q':
        text = extract_mdna_text(accession)
    elif filing_type == '8-K':
        text = extract_press_release(accession)

    if text:
        return text
    else:
        return pd.NA

async def check_compliance_async(session, row, sem, rate_lock, rate_state, index):
    f_type = str(row['Filing Type']).upper()
    url = row['SEC Link']
    headers = {'User-Agent': 'SoftUni Student Project Kamen Dimitrov (kamendd@hotmail.com)'}

    # 10-Qs are instant and don't need the network
    if '10-Q' in f_type:
        return True
    
    # 8-Ks: Wait for your specific "timeslot" to avoid the 429 error
    async with sem:
        # --- Global rate limiting block ---
        async with rate_lock:
            loop = asyncio.get_running_loop()
            now = loop.time()

            min_interval = 0.12
            elapsed = now - rate_state['last_call']

            if elapsed < min_interval:
                await asyncio.sleep(min_interval - elapsed)

            rate_state['last_call'] = loop.time()
        # --- End rate limiting block ---

        try:
            async with session.get(url, headers=headers, timeout=15) as response:
                if response.status == 200:
                    text = await response.text()
                    return bool(re.search(r'Item\s*(?:&nbsp;)?\s*2\.02', text, re.IGNORECASE))
                else:
                    return f"Error {response.status}"
        except Exception as e:
            return f"Fail: {str(e)}"


async def run_compliance_batch(df):
    sem = asyncio.Semaphore(9)
    connector = aiohttp.TCPConnector(limit=9)

    rate_lock = asyncio.Lock()
    rate_state = {'last_call': 0.0}

    data_list = df.to_dict('records')

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        for i, row in enumerate(data_list):
            tasks.append(
                check_compliance_async(session, row, sem, rate_lock, rate_state, i)
            )

        print("Starting SEC Compliance Check... this will take a few minutes.")
        results = await asyncio.gather(*tasks)
        return results

async def fetch_filing_content(session, row, sem, rate_lock, rate_state):
    headers = {
        'User-Agent': 'SoftUni Student Project Kamen Dimitrov (kamendd@hotmail.com)'
    }
    f_type = str(row['Filing Type']).upper()
    index_url = row['SEC Link']

    async with sem:
        async with rate_lock:
            loop = asyncio.get_running_loop()
            now = loop.time()
            min_interval = 0.11
            elapsed = now - rate_state['last_call']

            if elapsed < min_interval:
                await asyncio.sleep(min_interval - elapsed)

            rate_state['last_call'] = loop.time()

        try:
            async with session.get(index_url, headers=headers, timeout=20) as resp:
                if resp.status != 200:
                    return f"Index Error: {resp.status}"

                index_html = await resp.text()
                soup = BeautifulSoup(index_html, 'html.parser')

                table = soup.find('table', {'class': 'tableFile', 'summary': 'Document Format Files'})
                if not table:
                    return "Document Table Not Found"

                target_url = None

                for tr in table.find_all('tr')[1:]:
                    tds = tr.find_all('td')
                    if len(tds) < 4:
                        continue

                    doc_type = tds[3].get_text(strip=True)
                    doc_link_node = tds[2].find('a')

                    if not doc_link_node:
                        continue

                    if '8-K' in f_type and 'EX-99.1' in doc_type:
                        filename = doc_link_node['href'].split("/")[-1]
                        target_url = build_sec_document_url(index_url, filename)
                        break
                    elif '10-Q' in f_type and doc_type == '10-Q':
                        filename = doc_link_node['href'].split("/")[-1]
                        target_url = build_sec_document_url(index_url, filename)
                        break

                if not target_url:
                    return f"Specific {f_type} file link not found in table"

            async with rate_lock:
                loop = asyncio.get_running_loop()
                now = loop.time()
                min_interval = 0.11
                elapsed = now - rate_state['last_call']

                if elapsed < min_interval:
                    await asyncio.sleep(min_interval - elapsed)

                rate_state['last_call'] = loop.time()

            async with session.get(target_url, headers=headers, timeout=30) as content_resp:
                if content_resp.status != 200:
                    return f"Content Error: {content_resp.status}"

                raw_html = await content_resp.text()
                plain_text = html_to_text(raw_html)

                if '8-K' in f_type:
                    return plain_text

                elif '10-Q' in f_type:
                    return extract_mda_section(plain_text)

                return plain_text

        except Exception as e:
            return f"Scrape Error: {str(e)}"

async def run_extraction(df):
    sem = asyncio.Semaphore(8)
    connector = aiohttp.TCPConnector(limit_per_host=8)

    rate_lock = asyncio.Lock()
    rate_state = {'last_call': 0.0}

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            fetch_filing_content(session, row, sem, rate_lock, rate_state)
            for _, row in df.iterrows()
        ]
        return await asyncio.gather(*tasks)

# async def fetch_filing_content(session, row, sem):
#     headers = {'User-Agent': 'SoftUni Student Project Kamen Dimitrov (kamendd@hotmail.com)'}
#     f_type = str(row['Filing Type']).upper()
#     index_url = row['SEC Link'] # This is your "...-index.htm" link
    
#     async with sem:
#         try:
#             async with session.get(index_url, headers=headers) as resp:
#                 if resp.status != 200: return f"Index Error: {resp.status}"
#                 index_html = await resp.text()
#                 soup = BeautifulSoup(index_html, 'html.parser')
                
#                 # 1. Find the table containing the file list
#                 table = soup.find('table', {'class': 'tableFile', 'summary': 'Document Format Files'})
#                 if not table: return "Document Table Not Found"
                
#                 target_url = None
                
#                 # 2. Iterate through rows to find the right Document Type
#                 for tr in table.find_all('tr')[1:]: # Skip header
#                     tds = tr.find_all('td')
#                     if len(tds) < 4: continue
                    
#                     doc_type = tds[3].get_text(strip=True)
#                     doc_link_node = tds[2].find('a')
                    
#                     if '8-K' in f_type and 'EX-99.1' in doc_type:
#                         target_url = urljoin(index_url, doc_link_node['href'])
#                         break
#                     elif '10-Q' in f_type and '10-Q' in doc_type:
#                         target_url = urljoin(index_url, doc_link_node['href'])
#                         break
                
#                 if not target_url: return f"Specific {f_type} file link not found in table"

#                 # 3. Fetch the actual content from the discovered link
#                 async with session.get(target_url, headers=headers) as content_resp:
#                     return await content_resp.text()

#         except Exception as e:
#             return f"Scrape Error: {str(e)}"
            
# async def run_extraction(df):
#     sem = asyncio.Semaphore(10) # SEC Rate Limit
#     connector = aiohttp.TCPConnector(limit_per_host=10)
    
#     async with aiohttp.ClientSession(connector=connector) as session:
#         tasks = [fetch_filing_content(session, row, sem) for _, row in df.iterrows()]
#         return await asyncio.gather(*tasks)

#### 3. Establish the logic of verifying we have the needed 8-K filings for the scraping operations

The cell below illustrates the logic used behined the first step of extracting the actual text data I need It is to check if have filtered correctly 8-K filings, which are indeed earnings announcements by containing item 2.02 under items. The EDGAR Requests will take a long time (close to half hour for the filtered Data Set) if I choose a standard sequential crawl with a 0.11s delay specifically to adhere to the SEC's Fair Access Policy. However, it looks like extracting data from SEC's EDGAR database has become a branch of Data Science on its own foundations. As many people have already sought to time-optimize data handling performance at the edge of the SEC limit of ten requests per second, I took ready examples of code snippets to assemble asynchronous methods with semaphor system and multiple workers to optimize time for data handling without being throtled down, receiving 429 responses or being outright banned by the SEC. When using the code please define your own user agent to send as a header to SEC's EDGAR database.

In [4]:
test_url = "https://www.sec.gov/Archives/edgar/data/1341439/000119312526100148/0001193125-26-100148-index.htm"
headers = {'User-Agent': 'Kamen Dimitrov (kamendd@hotmail.com)'}

response = requests.get(test_url, headers=headers)
print(f"Status: {response.status_code}")

# Use a more robust regex that ignores extra spaces and HTML entities
pattern = r"Item\s*2\.02" 
match = re.search(pattern, response.text, re.IGNORECASE)

print(f"Match Found: {bool(match)}")

Status: 200
Match Found: True


#### 4. Run the first scraping operation to confirm we have the correct 8-K reports to finish this filtering data handling operation

Here is the actual data pipeline call, which implements the above illustration to all cells and thanks to the asynchronous capabilties developed in python, the particular Data Set will require 991 calls to the database, which are resolved in approximatelt 2 minutes. My first sequantial method took over 27 minutes, which prompted me to reasearch the async functionality (I have low familiarity with it), to take code snippets and finally ask the LLM's to iron out the bugs. 

In [17]:
#1 Take filings only for the last 10 years
filing_data = filing_data[filing_data['Filing Date'] >= '2016-03-01'].reset_index(drop=True)

#2 We are looking only for 8-K Earnings and 10-Q quarterly financial statements
filing_data['is_applicable_type'] = filing_data['Filing Type'].str.contains('8-K|10-Q', case=False, na=False)
filing_data = filing_data[filing_data['is_applicable_type'] == True].drop(columns=['is_applicable_type'])

#3 The Calcbench raw data contains True in 'Data' Field if it is a 8-K earnings
filing_data = filing_data[filing_data['Data'] == True].copy()

#4 We are down to 1891 rows. We need to scrape the EDGAR database to ascertain that the Items field in this filing contains 2.02,
#which means that is what we need and it will contain a press-release under EX-99.1
filing_data['is_verified_signal'] = pd.NA
print(f"Total rows: {len(filing_data)}")
print(f"Number of 8-Ks: {len(filing_data[filing_data['Filing Type'] == '8-K'])}")

#The code below is expected to run around 2 minutes of runtime
results = await run_compliance_batch(filing_data)
filing_data['is_verified_signal'] = results

filing_data

Total rows: 1894
Number of 8-Ks: 991
Starting SEC Compliance Check... this will take a few minutes.


,Filing ID,Company,Ticker,CIK,Filing Date,Filing Type,Period End,Fiscal Year,Fiscal Period,SEC Link,Calcbench Published,Data,is_verified_signal
0,2568709,ORACLE CORP,ORCL,1341439,2026-03-10,10-Q,27.2.2026,2026.0,3.0,https://www.sec.gov/Archives/edgar/data/134143...,2026-03-11,True,True
1,2563404,DEERE & CO,DE,315189,2026-02-25,10-Q,31.1.2026,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/315189...,2026-02-26,True,True
2,2562654,NVIDIA CORP,NVDA,1045810,2026-02-24,8-K,24.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104581...,2026-02-25,True,True
3,2562300,"Salesforce, Inc.",CRM,1108524,2026-02-24,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/110852...,2026-02-25,True,True
4,2559386,Walmart Inc.,WMT,104169,2026-02-18,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104169...,2026-02-19,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1889,350128,JPMORGAN CHASE & CO,JPM,19617,2016-04-12,8-K,30.3.2016,2016.0,1.0,http://www.sec.gov/Archives/edgar/data/19617/0...,2016-04-13,True,True
1890,349344,NIKE INC,NKE,320187,2016-04-05,10-Q,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/320187/...,2016-04-05,True,True
1891,346950,NIKE INC,NKE,320187,2016-03-21,8-K,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/320187/...,2016-03-22,True,True
1892,346679,ORACLE CORP,ORCL,1341439,2016-03-18,10-Q,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/1341439...,2016-03-18,True,True


#### 5. Let's save the result in a CSV file, which I will leave in static data

In [19]:
# 1. Drop the remaning N/A values where Exhibit 99.1 was not identified.
filing_data = filing_data[filing_data['is_verified_signal'] == True].copy()

# 2. Establish the number of datapoints in the final sample. I remain with 1879 rows for my 30 tickers.
print(filing_data['is_verified_signal'].value_counts())

# 3. Save the data to a CSV file
filing_data.to_csv("static_data/filing_data_verified_8_K.csv", index=False)

is_verified_signal
True    1879
Name: count, dtype: int64


In [28]:
verified_filing_data = pd.read_csv("static_data/filing_data_verified_8_K.csv")
verified_filing_data

,Filing ID,Company,Ticker,CIK,Filing Date,Filing Type,Period End,Fiscal Year,Fiscal Period,SEC Link,Calcbench Published,Data,is_verified_signal
0,2568709,ORACLE CORP,ORCL,1341439,2026-03-10,10-Q,27.2.2026,2026.0,3.0,https://www.sec.gov/Archives/edgar/data/134143...,2026-03-11,True,True
1,2563404,DEERE & CO,DE,315189,2026-02-25,10-Q,31.1.2026,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/315189...,2026-02-26,True,True
2,2562654,NVIDIA CORP,NVDA,1045810,2026-02-24,8-K,24.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104581...,2026-02-25,True,True
3,2562300,"Salesforce, Inc.",CRM,1108524,2026-02-24,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/110852...,2026-02-25,True,True
4,2559386,Walmart Inc.,WMT,104169,2026-02-18,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104169...,2026-02-19,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1874,350128,JPMORGAN CHASE & CO,JPM,19617,2016-04-12,8-K,30.3.2016,2016.0,1.0,http://www.sec.gov/Archives/edgar/data/19617/0...,2016-04-13,True,True
1875,349344,NIKE INC,NKE,320187,2016-04-05,10-Q,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/320187/...,2016-04-05,True,True
1876,346950,NIKE INC,NKE,320187,2016-03-21,8-K,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/320187/...,2016-03-22,True,True
1877,346679,ORACLE CORP,ORCL,1341439,2016-03-18,10-Q,28.2.2016,2016.0,3.0,http://www.sec.gov/Archives/edgar/data/1341439...,2016-03-18,True,True


In [35]:
test_sample = pd.concat([
    verified_filing_data[verified_filing_data['Filing Type'] == '10-Q'].head(5),
    verified_filing_data[verified_filing_data['Filing Type'] == '8-K'].head(5)
]).reset_index(drop=True)

test_sample['accession'] = test_sample['SEC Link'].apply(extract_accession_number)

start_time = time.perf_counter()
test_sample['text_output'] = test_sample.apply(process_filing_row, axis=1)
end_time = time.perf_counter()
duration = end_time - start_time
print(f"Operation completed in {duration:.4f} seconds")

test_sample

Operation completed in 126.0564 seconds


,Filing ID,Company,Ticker,CIK,Filing Date,Filing Type,Period End,Fiscal Year,Fiscal Period,SEC Link,Calcbench Published,Data,is_verified_signal,accession,text_output
0,2568709,ORACLE CORP,ORCL,1341439,2026-03-10,10-Q,27.2.2026,2026.0,3.0,https://www.sec.gov/Archives/edgar/data/134143...,2026-03-11,True,True,0001193125-26-101045,Item 2. \tManagement’s Discussion and Analysis...
1,2563404,DEERE & CO,DE,315189,2026-02-25,10-Q,31.1.2026,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/315189...,2026-02-26,True,True,0001104659-26-020158,Item 2. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
2,2552121,Walt Disney Co,DIS,1744489,2026-02-01,10-Q,26.12.2025,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/174448...,2026-02-02,True,True,0001744489-26-000019,MANAGEMENT’S DISCUSSION AND ANALYSIS OF\n\nFIN...
3,2551537,Apple Inc.,AAPL,320193,2026-01-29,10-Q,26.12.2025,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/320193...,2026-01-30,True,True,0000320193-26-000006,Item 2. Management’s Discussion and Analysi...
4,2551531,VISA INC.,V,1403161,2026-01-29,10-Q,30.12.2025,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/140316...,2026-01-30,True,True,0001403161-26-000045,Table of Contents\n\nITEM 2.Management’s Discu...
5,2562654,NVIDIA CORP,NVDA,1045810,2026-02-24,8-K,24.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104581...,2026-02-25,True,True,0001045810-26-000019,Document\n\nNVIDIA Announces Financial Results...
6,2562300,"Salesforce, Inc.",CRM,1108524,2026-02-24,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/110852...,2026-02-25,True,True,0001108524-26-000056,Document\n\nExhibit 99.1\n\nSalesforce Deliver...
7,2559386,Walmart Inc.,WMT,104169,2026-02-18,8-K,30.1.2026,2026.0,4.0,https://www.sec.gov/Archives/edgar/data/104169...,2026-02-19,True,True,0000104169-26-000032,Document\n\n\n ...
8,2559317,DEERE & CO,DE,315189,2026-02-18,8-K,31.1.2026,2026.0,1.0,https://www.sec.gov/Archives/edgar/data/315189...,2026-02-19,True,True,0001104659-26-017239,Exhibit 99.1\n(Furnished herewith)\n​\n\n\nNew...
9,2558457,FIRSTENERGY CORP,FE,1031296,2026-02-16,8-K,30.12.2025,2025.0,4.0,https://www.sec.gov/Archives/edgar/data/103129...,2026-02-17,True,True,0001031296-26-000041,Document\n\nExhibit 99.1\n\n\nFirstEnergy Corp...


In [36]:
test_sample['text_output'].str.len()

0     84701
1      7851
2     41492
3     17841
4     27532
5     60163
6     81024
7    112958
8     79114
9     28913
Name: text_output, dtype: int64

In [37]:
test_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Filing ID            10 non-null     int64  
 1   Company              10 non-null     object 
 2   Ticker               10 non-null     object 
 3   CIK                  10 non-null     int64  
 4   Filing Date          10 non-null     object 
 5   Filing Type          10 non-null     object 
 6   Period End           10 non-null     object 
 7   Fiscal Year          10 non-null     float64
 8   Fiscal Period        10 non-null     float64
 9   SEC Link             10 non-null     object 
 10  Calcbench Published  10 non-null     object 
 11  Data                 10 non-null     bool   
 12  is_verified_signal   10 non-null     bool   
 13  accession            10 non-null     object 
 14  text_output          10 non-null     object 
dtypes: bool(2), float64(2), int64(2), object(9)

#### 6. NLP text processing Source

@article{araci2019finbert,
  title={FinBERT: Financial Sentiment Analysis with Pre-trained Language Models},
  author={Araci, Dogu},
  journal={arXiv preprint arXiv:1908.10063},
  year={2019}
}

Hugging Face / ProsusAI Reference:

ProsusAI. (2020). FinBERT: A Pre-trained Strategy for Financial NLP. Hugging Face Model Hub. https://huggingface.co/ProsusAI/finbert

#### 7. Methodology & Model Choice
To analyze the "Hard Data" provided in this repository, I utilized the **FinBERT** (`ProsusAI/finbert`) model—a pre-trained Natural Language Processing (NLP) model specifically fine-tuned for financial sentiment.

* **Task:** Classification of around 450 unique transcripts (cleaned from an original 18,755 row dataset).
* **Processing:** Each transcript (avg. 7,333 tokens) will be split into overlapping 512-token chunks.
* **Aggregation:** The final score will be the mean average of all chunks within a single transcript, providing a global sentiment value between -1 (Bearish) and 1 (Bullish).

The cell below will provide an example of what I intend to do.

##### 7.1. Computational Hardware & Performance
Deep learning on large text blocks is computationally expensive. The results provided here will be generated using **GPU Acceleration** to ensure efficiency.

##### 7.2. **Benchmark Comparison**
If you choose to recalculate these scores using the provided script, please be aware of the following estimated processing times:

| Environment | Device Type | Avg. Time per Score | Total Time (280) |
| :--- | :--- | :--- | :--- |
| **Standard Laptop** | Intel/AMD CPU | ~33.3 seconds | **~3.5 to 4 Hours** |
| **High-End PC** | NVIDIA GPU (CUDA) | ~0.5 seconds | **~5 to 7 Minutes** |
| **Cloud (Google Colab)** | Tesla T4 GPU | ~0.4 seconds | **~4 Minutes** |

> **Instruction:** If running on a "regular" machine (CPU), ensure the laptop is plugged into a power source and "Sleep Mode" is disabled. The fans will likely run at maximum speed during this process.

Let's see the example cell:

In [ ]:
# # model_name = "ProsusAI/finbert"
# tokenizer = BertTokenizer.from_pretrained(model_name)
# model = BertForSequenceClassification.from_pretrained(model_name)

# device = 0 if torch.cuda.is_available() else -1 
# nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

# sample_text = transcript_data.iloc[0]['transcript']
# sentiment_results = nlp_utils.classify_long_transcript(sample_text, tokenizer, nlp)
# print(sentiment_results)

# final_score = nlp_utils.aggregate_sentiment(sentiment_results)
# print(f"Aggregate Sentiment Score: {final_score:.4f}")

#### 8. Let's now see what reports are available for different stocks.

I ran the following command in Google Colab on a T4 GPU runtime environment, where the calculation was done in under 4 minutes despite having to make over 2000 NLP calls. I will leave the code commented, while the calculated data is available with the repo.

In [ ]:
# final_dataset['sentiment_score'] = final_dataset['transcript'].apply(
#     lambda x: aggregate_sentiment(classify_long_transcript(x))
# )
# final_dataset.to_csv('final_transcript_dataset_with_sentiment.csv')

I would like to obtain industry and sector information from yfinance for each represented stock

In [ ]:
# sector_data = [data_pipe.get_sector_industry(t) for t in top_tickers]
# sector_df = pd.DataFrame(sector_data)
# sector_df

In [ ]:
# final_dataset_sentiment = final_dataset_sentiment.merge(sector_df, on='ticker', how='left')
# final_dataset_sentiment.to_csv("static_data/final_transcript_dataset_with_sentiment.csv", index=False)
# final_dataset_sentiment